In [1]:
# imports

import json
import os
from dotenv import load_dotenv
from openai import OpenAI

import anthropic
from IPython.display import Markdown, display, update_display
import google.generativeai


In [2]:
# Load environment variables in a file called .env
# Print the key prefixes to help with any debugging

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-
Google API Key exists and begins AIzaSyBu


In [3]:
# Connect to OpenAI, Anthropic

openai = OpenAI()
claude = anthropic.Anthropic()

In [4]:
# Gemini setup

google.generativeai.configure()
googleAI = OpenAI(
    api_key=google_api_key, 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)


In [5]:
JUDGE_SYSTEM_PROMPT = """You are an expert clinical educator evaluating simulated emergency department conversations.

You MUST respond with valid JSON only.
Do not include markdown, explanations, or extra text.
Do not wrap the JSON in code fences.

Evaluate realism, role fidelity, and educational suitability.
Be conservative in your ratings.
"""


In [6]:
JUDGE_USER_PROMPT = """
Conversation:
----------------
{FULL_CONVERSATION_TEXT}
----------------

Please rate the conversation using the following scale:

1 = Strongly disagree
2 = Disagree
3 = Neutral
4 = Agree
5 = Strongly agree

Return your answer in JSON with EXACTLY this structure:

{{
  "role_fidelity": 1-5,
  "turn_coherence": 1-5,
  "communication_realism": 1-5,
  "educational_usable": true/false,
  "comments": "string"
}}
"""



In [7]:
def evaluate_conversation_automatically(conversation_text, model="gpt-4o"):
    messages = [
        {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
        {
            "role": "user",
            "content": JUDGE_USER_PROMPT.format(
                FULL_CONVERSATION_TEXT=conversation_text
            )
        }
    ]

    response = openai.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0
    )

    return json.loads(response.choices[0].message.content)



In [45]:
## only for a single file 

conversations = []

with open("simulations.jsonl", "r", encoding="utf-8") as fh:
    for line in fh:
        conversations.append(json.loads(line))



In [46]:
## only for a single file 

with open("data/conversations/conv_73cf2ae2.json", "r", encoding="utf-8") as fh:
    convo = json.load(fh)


In [11]:
# convo.keys()


In [48]:
# only for  a single file  
conversation_text = "\n".join(
    f"{t['speaker'].upper()}: {t['text']}"
    for t in convo["turns"]
)



In [49]:
conversation_text = conversation_to_text(convo)
print(conversation_text[:1000])  # sanity check


DOCTOR: John, I'm the doctor looking after you today. Can you tell me what's happening?

PATIENT: I… I just have this awful chest pain.

NURSE: I'll start getting vitals now, doctor.

DOCTOR: Thank you. John, can you describe the chest pain for me? Is it sharp, dull, or tight? Does it radiate anywhere, like to your arm or jaw?

PATIENT: *breathing heavily*

It's... it's tight, like someone's got a vice around my chest. Started about an hour ago, just came on sudden-like. 

*clutches left arm*

Yeah, it's going down my left arm too. And my jaw's sore as well. I feel a bit sick, to be honest with you. Never had anything like this before. I'm... I'm worried, doctor.

NURSE: BP is 160/90, heart rate 110 and irregular, O2 sat 92% on room air. He's pale and diaphoretic, doctor. I'm putting him on the monitor and applying oxygen.

DOCTOR: Thank you for that information. John, I understand this is very concerning for you. We're going to take good care of you. 

I need to ask a few more questio

In [50]:
judge_result = evaluate_conversation_automatically(conversation_text)
print(judge_result)



{'role_fidelity': 4, 'turn_coherence': 4, 'communication_realism': 4, 'educational_usable': True, 'comments': 'The conversation demonstrates good role fidelity, with the doctor and nurse performing their roles appropriately, though there was a minor role violation with the patient initially. The turn coherence is strong, with logical progression and responses that align with the scenario. Communication realism is high, capturing the urgency and emotional weight of a potential heart attack situation. The scenario is educationally useful for demonstrating effective communication and management in an emergency setting, though the role violation could be addressed for improved fidelity.'}


In [ ]:
############################# batch files ########################

In [8]:
def conversation_json_to_text(convo_json):
    """
    Converts a saved conversation JSON into a readable transcript
    suitable for LLM judging.
    """
    lines = []
    for turn in convo_json["turns"]:
        speaker = turn["speaker"].upper()
        text = turn["text"].strip()
        lines.append(f"{speaker}: {text}")
    return "\n\n".join(lines)


In [9]:
from pathlib import Path

def evaluate_conversation_batch(
    conversations_dir,
    output_path="data/judge_results.jsonl",
    model="gpt-4o"
):
    """
    Evaluates all conversation JSON files in a directory using the LLM judge.

    Parameters
    ----------
    conversations_dir : str or Path
        Folder containing conversation JSON files
    output_path : str
        Where to append judge results (JSONL)
    model : str
        LLM model to use as judge
    """

    conversations_dir = Path(conversations_dir)
    results = []

    for json_file in conversations_dir.glob("*.json"):
        with open(json_file, "r", encoding="utf-8") as fh:
            convo = json.load(fh)

        conversation_text = conversation_json_to_text(convo)

        try:
            judge_result = evaluate_conversation_automatically(
                conversation_text,
                model=model
            )

            record = {
                "conversation_id": convo["conversation_id"],
                "file": json_file.name,
                "num_turns": convo["num_turns"],
                "role_guard_failures": convo["role_guard_failures"],
                "judge": judge_result
            }

            results.append(record)

            # append immediately (safe for long runs)
            with open(output_path, "a", encoding="utf-8") as out:
                out.write(json.dumps(record, ensure_ascii=False) + "\n")

            print(f"✓ Evaluated {json_file.name}")

        except Exception as e:
            print(f"✗ Failed on {json_file.name}: {e}")

    return results


In [12]:
results = evaluate_conversation_batch(
    conversations_dir="data/conversations",
    output_path="data/judge_results.jsonl",
    model="gpt-4o"
)


✓ Evaluated conv_02263e1d.json
✓ Evaluated conv_07aef0b9.json
✓ Evaluated conv_0e3ec2e6.json
✓ Evaluated conv_0f1141ee.json
✓ Evaluated conv_10a6c7dc.json
✓ Evaluated conv_2bf529b2.json
✓ Evaluated conv_30300709.json
✓ Evaluated conv_3c5a1b05.json
✓ Evaluated conv_46b0e3b2.json
✓ Evaluated conv_46defe37.json
✓ Evaluated conv_4a73a2e8.json
✓ Evaluated conv_544e2322.json
✓ Evaluated conv_568cb6e9.json
✓ Evaluated conv_5c0c9b3f.json
✓ Evaluated conv_5e0b4438.json
✓ Evaluated conv_73cf2ae2.json
✓ Evaluated conv_7b0158e1.json
✓ Evaluated conv_7b2a03ea.json
✓ Evaluated conv_81c04993.json
✓ Evaluated conv_88881828.json
✓ Evaluated conv_990e2cac.json
✓ Evaluated conv_aadab8dc.json
✓ Evaluated conv_af2283d0.json
✓ Evaluated conv_b306aeb3.json
✓ Evaluated conv_be0a4281.json
✓ Evaluated conv_c10efbc5.json
✓ Evaluated conv_e101cfd4.json
✓ Evaluated conv_f6002de1.json
✓ Evaluated conv_f98d056d.json


In [13]:
import os
print(os.getcwd())


C:\Users\Immersive.DESKTOP-NGU221C\projects\llm_engineering


In [14]:
print(os.listdir())


['.env', '.git', '.gitignore', '.ipynb_checkpoints', '.python-version', 'anaconda_projects', 'assets', 'community-contributions', 'data', 'environment.yml', 'extras', 'guides', 'images', 'jupyter_runtime', 'LICENSE', 'med_conv1.ipynb', 'med_conv1_orig.ipynb', 'med_convo_judge.ipynb', 'ODonovan_K92.pdf', 'output.md', 'output_markdown', 'pdf2markdown.ipynb', 'pyproject.toml', 'rag_markdown.ipynb', 'README.md', 'requirements.txt', 'resume.md', 'setup', 'simulations.jsonl', 'uv.lock', 'week1', 'week2', 'week3', 'week4', 'week5', 'week6', 'week7', 'week8']


In [15]:
print(os.listdir("data"))


['conversations', 'judge_results.jsonl', 'summaries.jsonl']


In [16]:
for root, dirs, files in os.walk(os.getcwd()):
    if "judge_results.jsonl" in files:
        print("FOUND:", os.path.join(root, "judge_results.jsonl"))


FOUND: C:\Users\Immersive.DESKTOP-NGU221C\projects\llm_engineering\data\judge_results.jsonl
